# RappiPlus | Data Cleaning and Preparation

## 1. Project Overview

**Project:** RappiPlus — E-commerce Business Performance Analysis  
**Author:** Camila Leal Ledesma  
**Tools:** Python, Pandas, NumPy

### Objective

Clean and prepare the datasets for business analysis by addressing the data quality issues identified during the initial assessment.

### Approach

- Preserve the original datasets.
- Apply cleaning decisions based on the data quality assessment.
- Document relevant transformations.
- Validate the results before exporting the cleaned datasets.

In [2]:
# Import libraries
import pandas as pd
import numpy as np

In [3]:
# Load raw datasets
orders = pd.read_csv('../data/raw/rappiplus_orders_raw.csv')
catalog = pd.read_csv('../data/raw/rappiplus_catalog.csv')
marketing = pd.read_csv('../data/raw/rappiplus_marketing_spend.csv')
experiment = pd.read_csv('../data/raw/experiment_checkout_ui.csv')

## 2. Orders Data Cleaning

This section addresses duplicate records, inconsistent values, missing data, and data types identified during the initial quality assessment.

Cleaning decisions are documented to ensure reliable and reproducible business analysis.

In [4]:
# Create a working copy of the orders dataset
orders_clean = orders.copy()

In [5]:
# Remove exact duplicate rows
orders_clean = orders_clean.drop_duplicates().copy()

print('Original rows:', len(orders))
print('Rows after removing duplicates:', len(orders_clean))

Original rows: 25100
Rows after removing duplicates: 25000


In [6]:
# Standardize country names
orders_clean['pais'] = orders_clean['pais'].str.capitalize()

# Check standardized countries
orders_clean['pais'].value_counts(dropna=False)

pais
Mexico       8341
Colombia     8304
Argentina    8055
NaN           300
Name: count, dtype: int64

In [7]:
# Convert order dates to datetime
orders_clean['fecha_hora_pedido'] = pd.to_datetime(
    orders_clean['fecha_hora_pedido']
)

# Check the updated data type
orders_clean['fecha_hora_pedido'].dtype

dtype('<M8[us]')

In [8]:
# Create product category reference
product_categories = catalog.set_index('nombre_producto')['categoria_producto']

In [9]:
# Recover missing product categories
orders_clean['categoria_producto'] = orders_clean['categoria_producto'].fillna(
    orders_clean['nombre_producto'].map(product_categories)
)

# Check remaining missing categories
orders_clean['categoria_producto'].isna().sum()

np.int64(30)

In [10]:
# Standardize product categories
orders_clean['categoria_producto'] = orders_clean['categoria_producto'].replace(
    {'Electronica': 'Electrónica'}
)

# Check standardized categories
orders_clean['categoria_producto'].value_counts(dropna=False)

categoria_producto
Hogar          8355
Moda           8324
Electrónica    8291
NaN              30
Name: count, dtype: int64

In [11]:
# Inspect orders with incomplete transaction details

incomplete_orders = orders_clean[
    orders_clean['cantidad'].isna()
]

incomplete_orders[
    [
        'id_pedido',
        'nombre_producto',
        'cantidad',
        'precio_unitario',
        'monto_descuento',
        'monto_total'
    ]
].head(10)

,id_pedido,nombre_producto,cantidad,precio_unitario,monto_descuento,monto_total
74,order_74,Sneakers-Urban-42,NaN,NaN,NaN,595.85
75,order_75,Sneakers-Urban-42,NaN,NaN,NaN,458.15
76,order_76,Laptop-Gaming-16GB,NaN,NaN,NaN,319.75
77,order_77,Vacuum-Pro-Black,NaN,NaN,NaN,227.55
78,order_78,Phone-Pro-128GB,NaN,NaN,NaN,432.39
79,order_79,Sneakers-Urban-42,NaN,NaN,NaN,527.15
80,order_80,Vacuum-Pro-Black,NaN,NaN,NaN,711.18
81,order_81,Tablet-Standard-64GB,NaN,NaN,NaN,41.08
82,order_82,Sneakers-Urban-42,NaN,NaN,NaN,431.55
83,order_83,Sneakers-Urban-42,NaN,NaN,NaN,279.35


In [12]:
# Assess the financial impact of incomplete orders

print('Incomplete orders:', len(incomplete_orders))

print(
    'Recorded amount from incomplete orders:',
    round(incomplete_orders['monto_total'].sum(), 2)
)

print(
    'Share of recorded amount (%):',
    round(
        incomplete_orders['monto_total'].sum()
        / orders_clean['monto_total'].sum() * 100,
        2
    )
)

Incomplete orders: 50
Recorded amount from incomplete orders: 21628.0
Share of recorded amount (%): 0.04


In [13]:
# Inspect orders with negative quantities

negative_orders = orders_clean[
    orders_clean['cantidad'] < 0
]

negative_orders[
    [
        'id_pedido',
        'nombre_producto',
        'cantidad',
        'precio_unitario',
        'monto_descuento',
        'monto_total'
    ]
]

,id_pedido,nombre_producto,cantidad,precio_unitario,monto_descuento,monto_total
266,order_266,Phone-Pro-128GB,-2.0,101.31,10.0,-192.62
267,order_267,Phone-Pro-128GB,-1.0,43.50,5.0,-38.50
268,order_268,Phone-Pro-128GB,-1.0,497.65,5.0,-492.65
269,order_269,Phone-Pro-128GB,-1.0,423.53,0.0,-423.53


In [14]:
# Assess the financial impact of negative quantities

print('Orders with negative quantities:', len(negative_orders))

print(
    'Total amount from negative quantities:',
    round(negative_orders['monto_total'].sum(), 2)
)

Orders with negative quantities: 4
Total amount from negative quantities: -1147.3


In [15]:
# Inspect orders with unusually high quantities

high_quantity_orders = orders_clean[
    orders_clean['cantidad'] > 100
]

high_quantity_orders[
    [
        'id_pedido',
        'nombre_producto',
        'cantidad',
        'precio_unitario',
        'monto_descuento',
        'monto_total'
    ]
]

,id_pedido,nombre_producto,cantidad,precio_unitario,monto_descuento,monto_total
3521,order_3521,Laptop-Gaming-16GB,10000.0,43.14,0.0,431400.0
3522,order_3522,Laptop-Gaming-16GB,10000.0,280.55,0.0,2805500.0
3586,order_3586,Laptop-Gaming-16GB,10000.0,490.35,0.0,4903500.0
3643,order_3643,Laptop-Gaming-16GB,10000.0,238.15,0.0,2381500.0
3656,order_3656,Laptop-Gaming-16GB,20000.0,297.66,0.0,5953200.0
3668,order_3668,Laptop-Gaming-16GB,20000.0,348.31,0.0,6966200.0
3689,order_3689,Laptop-Gaming-16GB,10000.0,87.69,0.0,876900.0
3722,order_3722,Laptop-Gaming-16GB,20000.0,442.01,0.0,8840200.0
3726,order_3726,Laptop-Gaming-16GB,20000.0,290.85,0.0,5817000.0
3748,order_3748,Laptop-Gaming-16GB,10000.0,336.93,0.0,3369300.0


In [16]:
# Check quantity distribution excluding extreme values

orders_clean.loc[
    orders_clean['cantidad'] <= 100,
    'cantidad'
].describe()

count    24940.000000
mean         1.504491
std          0.501191
min         -2.000000
25%          1.000000
50%          2.000000
75%          2.000000
max          2.000000
Name: cantidad, dtype: float64

In [17]:
# Assess the financial impact of extreme quantities

total_amount = orders_clean['monto_total'].sum()

extreme_amount = high_quantity_orders['monto_total'].sum()

amount_without_extremes = total_amount - extreme_amount

print('Total recorded amount:', round(total_amount, 2))
print('Amount from extreme orders:', round(extreme_amount, 2))
print('Amount excluding extremes:', round(amount_without_extremes, 2))

print(
    'Share of extreme orders (%):',
    round(extreme_amount / total_amount * 100, 2)
)

Total recorded amount: 51987462.26
Amount from extreme orders: 42344700.0
Amount excluding extremes: 9642762.26
Share of extreme orders (%): 81.45


In [18]:
# Separate extreme quantity orders

orders_outliers = orders_clean[
    orders_clean['cantidad'] > 100
].copy()

orders_clean = orders_clean[
    (orders_clean['cantidad'] <= 100)
    | (orders_clean['cantidad'].isna())
].copy()

print('Extreme orders separated:', len(orders_outliers))
print('Orders remaining:', len(orders_clean))

Extreme orders separated: 10
Orders remaining: 24990


In [19]:
# Validate orders after separating extreme quantities

print('Total orders:', len(orders_clean))

print('Missing quantities:', orders_clean['cantidad'].isna().sum())

print('Negative quantities:', (orders_clean['cantidad'] < 0).sum())

print('Quantities above 100:', (orders_clean['cantidad'] > 100).sum())

Total orders: 24990
Missing quantities: 50
Negative quantities: 4
Quantities above 100: 0


In [20]:
# Inspect orders with missing product information

missing_product_orders = orders_clean[
    orders_clean['nombre_producto'].isna()
]

missing_product_orders[
    [
        'id_pedido',
        'id_usuario',
        'fecha_hora_pedido',
        'nombre_producto',
        'categoria_producto',
        'cantidad',
        'precio_unitario',
        'monto_total'
    ]
].head(10)

,id_pedido,id_usuario,fecha_hora_pedido,nombre_producto,categoria_producto,cantidad,precio_unitario,monto_total
44,order_44,user_2899,2025-02-22,NaN,NaN,1.0,318.38,313.38
45,order_45,user_6196,2025-02-10,NaN,NaN,2.0,354.06,698.12
46,order_46,user_5815,2025-04-24,NaN,NaN,1.0,359.31,349.31
47,order_47,user_406,2025-05-08,NaN,NaN,2.0,476.78,953.56
48,order_48,user_6187,2025-01-26,NaN,NaN,2.0,137.96,275.93
49,order_49,user_7225,2025-04-04,NaN,NaN,2.0,433.76,867.53
50,order_50,user_1840,2025-02-05,NaN,NaN,1.0,495.38,490.38
51,order_51,user_5998,2025-02-21,NaN,NaN,2.0,382.94,765.89
52,order_52,user_5679,2025-04-17,NaN,NaN,2.0,415.01,830.01
53,order_53,user_5936,2025-01-02,NaN,NaN,2.0,236.49,472.98


In [21]:
# Check how many products share each selling price

price_products = orders_clean.groupby('precio_unitario')['nombre_producto'].nunique()

print('Prices linked to one product:', (price_products == 1).sum())
print('Prices linked to multiple products:', (price_products > 1).sum())

# Check missing-product orders with matching prices
missing_product_orders['precio_unitario'].isin(price_products.index).sum()

Prices linked to one product: 15515
Prices linked to multiple products: 4000


np.int64(30)

In [22]:
# Assess the impact of orders with missing products

print('Orders with missing products:', len(missing_product_orders))

print(
    'Recorded amount from missing-product orders:',
    round(missing_product_orders['monto_total'].sum(), 2)
)

print(
    'Share of recorded amount (%):',
    round(
        missing_product_orders['monto_total'].sum()
        / orders_clean['monto_total'].sum() * 100,
        2
    )
)

Orders with missing products: 30
Recorded amount from missing-product orders: 12262.62
Share of recorded amount (%): 0.13


In [23]:
# Check remaining missing values
orders_clean.isna().sum()

id_pedido               0
id_usuario              0
fecha_hora_pedido       0
pais                  300
dispositivo            20
fuente_referencia      30
nombre_producto        30
categoria_producto     30
cantidad               50
precio_unitario        50
monto_descuento        50
monto_total             0
dtype: int64

In [24]:
# Add unit costs from the product catalog

orders_clean = orders_clean.merge(
    catalog[['nombre_producto', 'costo_unitario']],
    on='nombre_producto',
    how='left',
    validate='many_to_one'
)

print('Total orders:', len(orders_clean))
print('Missing unit costs:', orders_clean['costo_unitario'].isna().sum())

Total orders: 24990
Missing unit costs: 30


In [25]:
# Calculate total product cost for positive-quantity orders

orders_clean['costo_total'] = np.where(
    orders_clean['cantidad'] > 0,
    orders_clean['cantidad'] * orders_clean['costo_unitario'],
    np.nan
)

# Check calculated and missing costs
print('Orders with calculated costs:', orders_clean['costo_total'].notna().sum())
print('Orders without calculated costs:', orders_clean['costo_total'].isna().sum())

Orders with calculated costs: 24906
Orders without calculated costs: 84


In [26]:
# Validate calculated product costs

valid_cost_orders = orders_clean[
    orders_clean['costo_total'].notna()
]

print('Orders with valid costs:', len(valid_cost_orders))

print(
    'Negative total costs:',
    (valid_cost_orders['costo_total'] < 0).sum()
)

print(
    'Zero total costs:',
    (valid_cost_orders['costo_total'] == 0).sum()
)

print(
    'Total calculated product cost:',
    round(valid_cost_orders['costo_total'].sum(), 2)
)

Orders with valid costs: 24906
Negative total costs: 0
Zero total costs: 0
Total calculated product cost: 3828869.01


In [27]:
# Identify orders eligible for profit calculation

orders_clean['profit_eligible'] = (
    (orders_clean['cantidad'] > 0)
    & orders_clean['costo_total'].notna()
    & orders_clean['monto_total'].notna()
)

print(
    'Orders eligible for profit calculation:',
    orders_clean['profit_eligible'].sum()
)

print(
    'Orders not eligible:',
    (~orders_clean['profit_eligible']).sum()
)

Orders eligible for profit calculation: 24906
Orders not eligible: 84


In [28]:
# Calculate estimated gross profit for eligible orders

orders_clean['utilidad_bruta'] = np.where(
    orders_clean['profit_eligible'],
    orders_clean['monto_total'] - orders_clean['costo_total'],
    np.nan
)

# Validate calculated profits
print(
    'Orders with calculated profit:',
    orders_clean['utilidad_bruta'].notna().sum()
)

print(
    'Orders without calculated profit:',
    orders_clean['utilidad_bruta'].isna().sum()
)

Orders with calculated profit: 24906
Orders without calculated profit: 84


In [29]:
# Check gross profit distribution

orders_clean['utilidad_bruta'].describe()

count    24906.000000
mean       232.118764
std        280.641535
min       -530.290000
25%         45.325000
50%        207.050000
75%        398.787500
max        979.140000
Name: utilidad_bruta, dtype: float64

In [30]:
# Check orders with negative gross profit

negative_profit_orders = orders_clean[
    orders_clean['utilidad_bruta'] < 0
]

print('Orders with negative profit:', len(negative_profit_orders))

print(
    'Products with negative profit:'
)

print(negative_profit_orders['nombre_producto'].value_counts())

Orders with negative profit: 4528
Products with negative profit:
nombre_producto
Jacket-Winter-M         1525
Laptop-Gaming-16GB      1498
Blender-XL-Red          1414
Tablet-Standard-64GB      48
Sneakers-Urban-42         31
Vacuum-Pro-Black          10
Phone-Pro-128GB            2
Name: count, dtype: int64


### Profit Calculation and Data Limitations

Gross profit was estimated using recorded sales amounts and product costs from the catalog.

- 24,906 orders have sufficient information for profit calculation.
- 84 orders remain without calculated profit due to missing transaction details or pending return treatment.
- 4,528 orders have negative estimated gross profit.
- Ten extreme-quantity orders were excluded from the main analytical dataset and preserved separately.

Missing costs were not replaced with zero.

Gross profit estimates exclude marketing expenses and other operating costs. Potential returns require separate treatment.

Product-level profitability will be investigated in the business analysis notebook.

In [31]:
# Classify transactions by quantity

orders_clean['tipo_transaccion'] = np.select(
    [
        orders_clean['cantidad'] < 0,
        orders_clean['cantidad'] > 0
    ],
    [
        'Possible return',
        'Sale'
    ],
    default='Unknown'
)

orders_clean['tipo_transaccion'].value_counts()

tipo_transaccion
Sale               24936
Unknown               50
Possible return        4
Name: count, dtype: int64

In [32]:
# Handle remaining missing categorical values

categorical_columns = [
    'pais',
    'dispositivo',
    'fuente_referencia',
    'nombre_producto',
    'categoria_producto'
]

orders_clean[categorical_columns] = (
    orders_clean[categorical_columns].fillna('Unknown')
)

# Validate remaining missing values
orders_clean.isna().sum()

id_pedido              0
id_usuario             0
fecha_hora_pedido      0
pais                   0
dispositivo            0
fuente_referencia      0
nombre_producto        0
categoria_producto     0
cantidad              50
precio_unitario       50
monto_descuento       50
monto_total            0
costo_unitario        30
costo_total           84
profit_eligible        0
utilidad_bruta        84
tipo_transaccion       0
dtype: int64

In [33]:
# Final validation of cleaned orders

print('Total orders:', len(orders_clean))
print('Duplicate order IDs:', orders_clean['id_pedido'].duplicated().sum())
print('Extreme quantities:', (orders_clean['cantidad'] > 100).sum())

print(
    'Orders with calculated profit:',
    orders_clean['utilidad_bruta'].notna().sum()
)

print(
    'Orders without calculated profit:',
    orders_clean['utilidad_bruta'].isna().sum()
)

print(
    'Total estimated gross profit:',
    round(orders_clean['utilidad_bruta'].sum(), 2)
)

Total orders: 24990
Duplicate order IDs: 0
Extreme quantities: 0
Orders with calculated profit: 24906
Orders without calculated profit: 84
Total estimated gross profit: 5781149.93


## 3. Marketing Data Cleaning

This section prepares marketing spending data for business analysis.

The initial assessment identified 101 missing channel values that can be recovered using campaign identifiers.

Dates will be converted to datetime, and the original dataset will be preserved.

In [34]:
# Create a working copy
marketing_clean = marketing.copy()

# Convert dates
marketing_clean['fecha'] = pd.to_datetime(
    marketing_clean['fecha']
)

# Recover missing channels using campaign IDs
campaign_channels = marketing_clean.groupby('id_campaña')['canal'].first()

marketing_clean['canal'] = marketing_clean['canal'].fillna(
    marketing_clean['id_campaña'].map(campaign_channels)
)

# Validate results
print('Total records:', len(marketing_clean))
print('Missing channels:', marketing_clean['canal'].isna().sum())
print('Duplicate records:', marketing_clean.duplicated().sum())
print('Date type:', marketing_clean['fecha'].dtype)

Total records: 1620
Missing channels: 0
Duplicate records: 0
Date type: datetime64[us]


## 4. Product Catalog Preparation

The product catalog contains seven unique products with complete information and valid unit costs.

No additional cleaning transformations are required.

In [35]:
# Prepare product catalog

catalog_clean = catalog.copy()

# Validate catalog
print('Total products:', len(catalog_clean))
print('Duplicate products:', catalog_clean['nombre_producto'].duplicated().sum())
print('Missing values:', catalog_clean.isna().sum().sum())
print('Invalid unit costs:', (catalog_clean['costo_unitario'] <= 0).sum())

Total products: 7
Duplicate products: 0
Missing values: 0
Invalid unit costs: 0


## 5. A/B Experiment Data Preparation

The experiment dataset contains 10,000 unique users with complete records and valid conversion values.

The preparation focuses on converting timestamps to datetime and preserving the original experiment data.

Statistical analysis will be performed separately.

In [36]:
# Prepare A/B experiment dataset

experiment_clean = experiment.copy()

# Convert timestamps
experiment_clean['timestamp'] = pd.to_datetime(
    experiment_clean['timestamp']
)

# Validate results
print('Total records:', len(experiment_clean))
print('Duplicate users:', experiment_clean['id_usuario'].duplicated().sum())
print('Missing values:', experiment_clean.isna().sum().sum())
print('Date type:', experiment_clean['timestamp'].dtype)
print('Variants:', experiment_clean['variante'].unique())
print('Conversion values:', experiment_clean['convirtio'].unique())

Total records: 10000
Duplicate users: 0
Missing values: 0
Date type: datetime64[us]
Variants: <StringArray>
['tratamiento', 'control']
Length: 2, dtype: str
Conversion values: [0 1]


In [37]:
# Export cleaned datasets

from pathlib import Path

output_dir = Path('../data/processed')
output_dir.mkdir(parents=True, exist_ok=True)

orders_clean.to_csv(output_dir / 'rappiplus_orders_clean.csv', index=False)
marketing_clean.to_csv(output_dir / 'rappiplus_marketing_clean.csv', index=False)
catalog_clean.to_csv(output_dir / 'rappiplus_catalog_clean.csv', index=False)
experiment_clean.to_csv(output_dir / 'rappiplus_experiment_clean.csv', index=False)

# Preserve excluded extreme orders for auditing
orders_outliers.to_csv(
    output_dir / 'rappiplus_orders_outliers.csv',
    index=False
)

print('Datasets exported successfully.')

Datasets exported successfully.


In [38]:
# Validate exported datasets

datasets = {
    'Orders': 'rappiplus_orders_clean.csv',
    'Marketing': 'rappiplus_marketing_clean.csv',
    'Catalog': 'rappiplus_catalog_clean.csv',
    'Experiment': 'rappiplus_experiment_clean.csv',
    'Outliers': 'rappiplus_orders_outliers.csv'
}

for name, filename in datasets.items():
    df = pd.read_csv(output_dir / filename)
    print(f'{name}: {len(df)} rows')

Orders: 24990 rows
Marketing: 1620 rows
Catalog: 7 rows
Experiment: 10000 rows
Outliers: 10 rows
